# Structured output

Models can be requested to provide their response in a format matching a given schema.This is useful for ensuring the output can be easilu parsed and used in subsequent procesing, LangChain supports multiple schema types and methods for enforcing structured output.

# Pydantic

Pydantic models provide the richest feature set with validation, description and nested structures.

In [2]:
import os
import time
from langchain_ollama import ChatOllama

model = ChatOllama(
    model='qwen2.5:3b',
    temperature=0
) 

res = model.stream('How are you?')
for chunk in res:
    print(chunk.content, end='', flush=True)
    time.sleep(0.3)
    

I'm just a computer program, so I don't have feelings. However, I'm here and ready to assist you. How can I help you today?

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str=Field(description="The title of movie")
    year: int=Field(description="This year the movie was released")
    directot: str=Field(description="The director of the movie")
    rating: float=Field(description="The movie rating out of 10")


In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6'}}, model='qwen2.5:3b', temperature=0.0), kwargs={'format': {'properties': {'title': {'description': 'The title of movie', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'title': 'Year', 'type': 'integer'}, 'directot': {'description': 'The director of the movie', 'title': 'Directot', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'title': 'Rating', 'type': 'number'}}, 'required': ['title', 'year', 'directot', 'rating'], 'title': 'Movie', 'type': 'object'}, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema'}, 'schema': <class '__main__.Movie'>}}, config={}, config_factories=[])
| PydanticOutputParser(pydantic_object=<class '__main__.Movie'>)

In [5]:
res = model.invoke("Provide the details about the movie Inception")
res

AIMessage(content='"Inception" is a science fiction film directed by Christopher Nolan and released in 2010. The movie stars Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page Evans, and Tom Hardy. Here are some key details about the movie:\n\n### Plot Summary\nThe film revolves around Dom Cobb (played by Leonardo DiCaprio), a skilled thief who can enter people\'s dreams to steal secrets. Cobb is a "sleeper cell," a term used by the movie to describe individuals who have been conditioned to forget their experiences in dreams. He is tasked with a dangerous mission: to enter the subconscious of a wealthy businessman named Robert Fischer (played by Joseph Gordon-Levitt) and extract information from his subconscious mind.\n\n### Key Characters\n- **Dom Cobb**: The main character, a skilled thief who can enter people\'s dreams.\n- **Robert Fischer**: The target of the heist, a wealthy businessman who is being blackmailed by his ex-wife.\n- **Arthur** (played by Tom Hardy): Cobb\'s partner 

In [6]:
res = model_with_structure.invoke("Provide the details about the movie Baghban ?")
res

Movie(title='Baghban', year=2006, directot='Shyam Benegal', rating=7.5)

### Message ouput alongside parsed structure

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str=Field(..., description="The title of movie")
    year: int=Field(..., description="This year the movie was released")
    directot: str=Field(..., description="The director of the movie")
    rating: float=Field(..., description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

res = model_with_structure.invoke('Provide the details about the movie Inception')
res

{'raw': AIMessage(content='{ "title": "Inception", "year": 2010, "directot": "Christopher Nolan", "rating": 8.8 }', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-26T09:21:10.696146Z', 'done': True, 'done_reason': 'stop', 'total_duration': 710099541, 'load_duration': 857500, 'prompt_eval_count': 37, 'prompt_eval_duration': 50969000, 'eval_count': 35, 'eval_duration': 655580000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a17783-8360-7116-ad6e-51fdb0a09b1a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 35, 'total_tokens': 72}),
 'parsed': Movie(title='Inception', year=2010, directot='Christopher Nolan', rating=8.8),
 'parsing_error': None}

### Nested structure

In [8]:
class Actor(BaseModel):
    name: str
    role:str


class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
res = model_with_structure.invoke('Provide the details about the movie Inception')
res


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Ellen Page', role='Artemis Cobb'), Actor(name='Joseph Gordon-Levitt', role="Arthur 'Arthur' Eames"), Actor(name='Tom Hardy', role='Preacher'), Actor(name='Michael Caine', role='Robert F. Fischer')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=140000000.00000003)

# TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [9]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[str, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_type_dict = model.with_structured_output(MovieDict)
model_with_type_dict.invoke("Please provide the details of the movie avengers")

{'title': 'Avengers', 'year': '', 'director': 'Joss Whedon', 'rating': 8.6}

In [14]:
class Actor(TypedDict):
    name: str
    role:str


class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]

model_with_structure = model.with_structured_output(MovieDetails)
res = model_with_structure.invoke('Provide the details about the movie Inception')
res

{'title': 'Inception',
 'year': 2010,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Ellen Page', 'role': 'Artemis Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': "Arthur 'Arthur' Eames"},
  {'name': 'Tom Hardy', 'role': 'Preacher'},
  {'name': 'Michael Caine', 'role': 'Robert F. Fischer'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Thriller']}

# DataClasses